# 가족실태조사 기반 공동참여도·계층인식 지표 3종 산출

## 1. 분석 목적과 산식

2020년과 2023년 가족실태조사 원자료를 연도별로 독립 처리하여 전국과 17개 시도의 다음 지표를 산출한다. 전국값은 시도값의 단순평균이 아니라 전국 원자료에서 직접 계산한다.

- **가사노동 공동 참여도(%)** = 가사노동 문항에서 `남편과 아내가 똑같이`라고 응답한 사람의 가구원 가중치 합 / 유효응답(1–3) 가구원 가중치 합 × 100
- **돌봄노동 공동 참여도(%)** = 자녀양육(과 교육) 문항에서 `남편과 아내가 똑같이`라고 응답한 사람의 가구원 가중치 합 / 유효응답(1–3) 가구원 가중치 합 × 100
- **사회경제적 지위에 대한 인식(점)** = 1–5점 응답의 가구원 가중평균

가사·돌봄은 소수점 첫째 자리, 계층 인식은 소수점 둘째 자리에서 최종 반올림한다. 계산 중에는 원래 정밀도를 유지하고 유효 가중분모가 0인 지역은 결측으로 둔다.

## 2. 분석 연도와 조사표 근거

2015년 자료는 공개 지역변수가 시도가 아니라 동부·읍면부 구분이고, 가사·돌봄 문항 정의도 2020·2023년의 실제 수행 주체 문항과 달라 제외한다. 다른 연도의 값으로 대체하거나 보간하지 않는다.

응답코드와 문항 분기는 다음 공식 자료와 실제 CSV 분포를 함께 대조했다.

- [국가데이터처 2020년 가족실태조사 조사표](https://mods.go.kr/board.es?act=view&bid=12073&list_no=416317&mid=b40102010200): 현재 배우자가 있는 가구원이 배우자 관계 영역에 응답하며, 21-1 수행 주체 코드는 남편 1, 아내 2, 똑같이 3, 자녀양육의 해당 없음 4로 제시된다.
- [성평등가족부 2023년 제5차 가족실태조사 연구](https://www.mogef.go.kr/mp/pcd/mp_pcd_s001d.do?bbtSn=704992&mid=plc503): 조사표 부록과 결과표에서 같은 수행 주체 체계, 계층 인식 1–5점 방향, 유배우자 대상 분기를 확인했다.

아래에서는 이 근거가 원자료의 결측·비해당 구조와 정확히 부합하는지 다시 검증한다.

## 3. 라이브러리와 경로 설정

In [ ]:
import os
from pathlib import Path
from numbers import Real

import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in (cwd, cwd.parent) if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
DEFAULT_RAW_DIR = REPO_ROOT / "data" / "raw" / "구조환경지수 원데이터 구축용" / "가족실태조사"
RAW_DIR = Path(os.environ.get("YUMOCHA_FAMILY_SURVEY_DIR", DEFAULT_RAW_DIR))
OUTPUT_DIR = REPO_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = {
    2020: RAW_DIR / "2020_총괄_20260726_88104.csv",
    2023: RAW_DIR / "2023_총괄_20260726_88104.csv",
}
OUTPUT_FILES = {
    "가사노동 공동 참여도": OUTPUT_DIR / "가족실태조사_가사노동_공동_참여도_2020_2023.csv",
    "돌봄노동 공동 참여도": OUTPUT_DIR / "가족실태조사_돌봄노동_공동_참여도_2020_2023.csv",
    "사회경제적 지위에 대한 인식": OUTPUT_DIR
    / "가족실태조사_사회경제적_지위에_대한_인식_2020_2023.csv",
}

print("저장소:", REPO_ROOT)
print("원자료:", RAW_DIR)
print("출력:", OUTPUT_DIR)

## 4. 2020년·2023년 CSV 식별과 로딩

확인된 파일명만 명시적으로 사용한다. 기본 원자료 경로는 저장소 내부 `data/raw/구조환경지수 원데이터 구축용/가족실태조사`이며, 별도 위치를 사용할 때는 환경변수 `YUMOCHA_FAMILY_SURVEY_DIR`로 지정한다. 인코딩은 파일 바이트가 오류 없이 완전히 디코딩되는 후보를 확인한 뒤 적용한다. 원자료 변경 여부를 마지막 QA에서 확인하기 위해 크기와 수정시각도 기록한다.

In [ ]:
def identify_encoding(path):
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "cp949"):
        try:
            raw.decode(encoding)
            return encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"지원 인코딩으로 읽을 수 없음: {path}")


for path in SOURCE_FILES.values():
    assert path.is_file(), f"원자료 없음: {path}"

source_signatures = {
    year: (path.stat().st_size, path.stat().st_mtime_ns) for year, path in SOURCE_FILES.items()
}
encodings = {year: identify_encoding(path) for year, path in SOURCE_FILES.items()}
raw_by_year = {
    year: pd.read_csv(path, encoding=encodings[year], dtype={"가구일련번호": "string"})
    for year, path in SOURCE_FILES.items()
}

file_summary = pd.DataFrame(
    [
        {
            "연도": year,
            "파일명": path.name,
            "인코딩": encodings[year],
            "행": raw_by_year[year].shape[0],
            "열": raw_by_year[year].shape[1],
        }
        for year, path in SOURCE_FILES.items()
    ]
)
display(file_summary)
for year, data in raw_by_year.items():
    print(f"[{year}] 전체 컬럼명: {list(data.columns)}")

## 5. 필요한 컬럼만 구조·응답코드 검증

실제 확인한 전체 컬럼명을 연도별 명시적 딕셔너리로 공통 별칭에 연결한다. 부분 문자열 검색이나 유사도 기반 자동 선택은 사용하지 않는다. `가구일련번호`는 중복 제거용이 아닌 검증용 식별자다.

In [ ]:
COLUMN_ALIASES = {
    2020: {
        "가구일련번호": "가구일련번호",
        "지역": "시도코드",
        "13) 전반적계층": "사회경제적지위",
        "14) 혼인상태": "혼인상태",
        "15) 자녀유무": "자녀유무",
        "21-1) 가사수행_주로누가_가사노동": "가사노동",
        "21-1) 가사수행_주로누가_자녀양육": "돌봄노동",
        "WT1_가구원가중치": "가구원가중치",
    },
    2023: {
        "가구일련번호": "가구일련번호",
        "지역": "시도코드",
        "12) 전반적계층": "사회경제적지위",
        "13) 혼인상태": "혼인상태",
        "14) 자녀유무": "자녀유무",
        "23-1) 가사수행_주로누가_가사노동": "가사노동",
        "23-1) 가사수행_주로누가_자녀양육과교육": "돌봄노동",
        "가구원 가중치": "가구원가중치",
    },
}
EXPECTED_SHAPES = {2020: (22173, 8), 2023: (24488, 8)}

data_by_year = {}
for year, raw in raw_by_year.items():
    assert raw.shape == EXPECTED_SHAPES[year]
    assert list(raw.columns) == list(COLUMN_ALIASES[year])
    data_by_year[year] = raw.rename(columns=COLUMN_ALIASES[year]).copy()

alias_table = pd.DataFrame(
    [
        {"연도": year, "원본 컬럼": original, "공통 별칭": alias}
        for year, mapping in COLUMN_ALIASES.items()
        for original, alias in mapping.items()
    ]
)
display(alias_table)

In [ ]:
def format_code(value):
    if pd.isna(value):
        return "<결측>"
    if isinstance(value, Real) and float(value).is_integer():
        return str(int(value))
    return str(value)


dtype_rows = []
distribution_rows = []
weight_rows = []
branch_rows = []
categorical_columns = ["시도코드", "사회경제적지위", "혼인상태", "자녀유무", "가사노동", "돌봄노동"]

for year, data in data_by_year.items():
    inverse_alias = {alias: original for original, alias in COLUMN_ALIASES[year].items()}
    for alias in data.columns:
        dtype_rows.append(
            {
                "연도": year,
                "공통 별칭": alias,
                "원본 컬럼": inverse_alias[alias],
                "자료형": str(data[alias].dtype),
                "결측": int(data[alias].isna().sum()),
            }
        )
    for column in categorical_columns:
        counts = data[column].value_counts(dropna=False).sort_index()
        distribution_rows.extend(
            {"연도": year, "변수": column, "응답코드": format_code(code), "건수": int(count)}
            for code, count in counts.items()
        )
    weight = data["가구원가중치"]
    weight_rows.append(
        {
            "연도": year,
            "자료형": str(weight.dtype),
            "결측": int(weight.isna().sum()),
            "0": int(weight.eq(0).sum()),
            "음수": int(weight.lt(0).sum()),
            "최솟값": weight.min(),
            "최댓값": weight.max(),
            "합계": weight.sum(),
        }
    )
    not_spouse = data["혼인상태"].ne(2)
    house_missing = data["가사노동"].isna()
    care_missing = data["돌봄노동"].isna()
    branch_rows.append(
        {
            "연도": year,
            "혼인상태≠2": int(not_spouse.sum()),
            "가사 결측": int(house_missing.sum()),
            "돌봄 결측": int(care_missing.sum()),
            "가사 분기 일치": bool(not_spouse.equals(house_missing)),
            "돌봄 분기 일치": bool(not_spouse.equals(care_missing)),
        }
    )

print("필수 컬럼 자료형과 결측")
display(pd.DataFrame(dtype_rows))
print("제한된 응답코드 분포")
display(pd.DataFrame(distribution_rows))
print("가구원 가중치 점검")
display(pd.DataFrame(weight_rows))
print("배우자 관계 문항 분기 점검")
display(pd.DataFrame(branch_rows))

for year, data in data_by_year.items():
    print(f"[{year}] 자녀유무 × 돌봄노동 응답코드")
    display(pd.crosstab(data["자녀유무"], data["돌봄노동"], dropna=False, margins=True))

## 6. 확정한 응답코드와 분모 처리

공식 조사표와 위 분포·교차표에 따라 다음과 같이 확정한다.

- 2020년 설계 응답코드: 가사노동은 `1–3`, 돌봄노동은 `1–4`이다.
- 2023년 설계 응답코드: 가사노동과 돌봄노동 모두 `1–4`이다. 다만 실제 2023년 가사노동 자료에서 코드 4는 0건이다.
- 수행 주체 코드는 `1=남편`, `2=아내`, `3=남편과 아내가 똑같이`, `4=해당 없음`이다. 지표 분모는 수행 주체 유효응답인 1–3으로 정의한다.
- 사회경제적 지위: `1=하층`, `2=중하층`, `3=중층`, `4=중상층`, `5=상층`. 높을수록 인식 지위가 높으므로 역코딩하지 않는다.
- 자녀유무: `1=있음`, `2=없음`.
- 혼인상태 2020: `1=미혼`, `2=배우자 있음(사실혼·비혼동거 포함)`, `3=별거 또는 이혼`, `4=사별`.
- 혼인상태 2023: `1=미혼`, `2=배우자 있음(사실혼·비혼동거 포함)`, `3=별거`, `4=이혼`, `5=사별`.

두 연도 모두 가사·돌봄 결측은 `혼인상태 != 2`와 정확히 일치하여 이 문항이 현재 배우자가 있는 응답자에게 적용됐음을 확인했다. 돌봄 코드 4는 자녀 없음뿐 아니라 자녀 있음에도 관측되므로, 자녀유무를 추가 필터로 중복 적용하지 않고 공식 수행 주체 코드 1–3을 분모로 삼는다. CSV에 별도의 무응답·모름 코드는 관측되지 않았으며 결측과 코드 4는 분모에서 제외한다.

## 7. 지역코드 표준화

In [ ]:
REGION_MAP = {
    11: "서울",
    21: "부산",
    22: "대구",
    23: "인천",
    24: "광주",
    25: "대전",
    26: "울산",
    29: "세종",
    31: "경기",
    32: "강원",
    33: "충북",
    34: "충남",
    35: "전북",
    36: "전남",
    37: "경북",
    38: "경남",
    39: "제주",
}
REGION_ORDER = ["전국", *REGION_MAP.values()]

region_check_rows = []
for year, data in data_by_year.items():
    observed = sorted(data["시도코드"].dropna().unique().tolist())
    assert observed == sorted(REGION_MAP)
    data["지역명"] = data["시도코드"].map(REGION_MAP)
    assert data["지역명"].notna().all()
    region_check_rows.append(
        {
            "연도": year,
            "지역코드 수": len(observed),
            "코드": observed,
            "17개 시도 식별": len(observed) == 17,
        }
    )

display(pd.DataFrame({"시도코드": REGION_MAP.keys(), "지역": REGION_MAP.values()}))
display(pd.DataFrame(region_check_rows))

## 8. 연도별 전국·시도 가중 지표 산출

전국은 각 연도 전체 원자료를 함수에 직접 전달한다. 가구일련번호 중복은 동일 가구의 복수 가구원을 뜻하므로 행을 제거하지 않는다. 모든 산식은 가구원 가중치를 사용한다.

In [ ]:
RESPONSE_CODE_DESIGN = {
    2020: {"가사노동": {1, 2, 3}, "돌봄노동": {1, 2, 3, 4}},
    2023: {"가사노동": {1, 2, 3, 4}, "돌봄노동": {1, 2, 3, 4}},
}
for year, data in data_by_year.items():
    for column, designed_codes in RESPONSE_CODE_DESIGN[year].items():
        observed_codes = set(data[column].dropna().astype(int).unique())
        assert observed_codes.issubset(designed_codes)
assert data_by_year[2023]["가사노동"].eq(4).sum() == 0

HOUSE_VALID = {1, 2, 3}
CARE_VALID = {1, 2, 3}
SES_VALID = {1, 2, 3, 4, 5}
EQUAL_CODE = 3


def weighted_equal_share(data, column, valid_codes):
    valid = data[column].isin(valid_codes)
    denominator = data.loc[valid, "가구원가중치"].sum()
    if denominator == 0:
        return np.nan, int(valid.sum()), denominator
    numerator = data.loc[valid & data[column].eq(EQUAL_CODE), "가구원가중치"].sum()
    return numerator / denominator * 100, int(valid.sum()), denominator


def weighted_mean_ses(data):
    valid = data["사회경제적지위"].isin(SES_VALID)
    denominator = data.loc[valid, "가구원가중치"].sum()
    if denominator == 0:
        return np.nan, int(valid.sum()), denominator
    numerator = (data.loc[valid, "사회경제적지위"] * data.loc[valid, "가구원가중치"]).sum()
    return numerator / denominator, int(valid.sum()), denominator


metric_names = [
    "가사노동 공동 참여도",
    "돌봄노동 공동 참여도",
    "사회경제적 지위에 대한 인식",
]
raw_values = {metric: {2020: {}, 2023: {}} for metric in metric_names}
diagnostic_rows = []

for year, data in data_by_year.items():
    groups = [("전국", data)] + [
        (region, data.loc[data["지역명"].eq(region)]) for region in REGION_MAP.values()
    ]
    for region, subset in groups:
        house_value, house_n, house_denominator = weighted_equal_share(
            subset, "가사노동", HOUSE_VALID
        )
        care_value, care_n, care_denominator = weighted_equal_share(subset, "돌봄노동", CARE_VALID)
        ses_value, ses_n, ses_denominator = weighted_mean_ses(subset)
        raw_values["가사노동 공동 참여도"][year][region] = house_value
        raw_values["돌봄노동 공동 참여도"][year][region] = care_value
        raw_values["사회경제적 지위에 대한 인식"][year][region] = ses_value
        diagnostic_rows.append(
            {
                "연도": year,
                "지역": region,
                "가사 유효응답수": house_n,
                "가사 가중분모": house_denominator,
                "돌봄 유효응답수": care_n,
                "돌봄 가중분모": care_denominator,
                "계층 유효응답수": ses_n,
                "계층 가중분모": ses_denominator,
            }
        )

diagnostics = pd.DataFrame(diagnostic_rows)
display(diagnostics)

## 9. 지표별 최종 데이터프레임

In [ ]:
ROUND_DIGITS = {
    "가사노동 공동 참여도": 1,
    "돌봄노동 공동 참여도": 1,
    "사회경제적 지위에 대한 인식": 2,
}

final_tables = {}
for metric in metric_names:
    digits = ROUND_DIGITS[metric]
    final_tables[metric] = pd.DataFrame(
        {
            "지역": REGION_ORDER,
            "세부지표": metric,
            "2020": [raw_values[metric][2020][region] for region in REGION_ORDER],
            "2023": [raw_values[metric][2023][region] for region in REGION_ORDER],
        }
    ).assign(
        **{
            "2020": lambda frame: frame["2020"].round(digits),
            "2023": lambda frame: frame["2023"].round(digits),
        }
    )

for metric, table in final_tables.items():
    assert table.shape == (18, 4)
    assert list(table.columns) == ["지역", "세부지표", "2020", "2023"]
    assert table["지역"].tolist() == REGION_ORDER
    print(metric, table.shape)
    display(table)

combined = pd.concat(final_tables.values(), ignore_index=True)
assert combined.shape == (54, 4)
assert (
    combined.loc[combined["세부지표"].isin(metric_names[:2]), ["2020", "2023"]]
    .stack()
    .between(0, 100)
    .all()
)
assert (
    combined.loc[combined["세부지표"].eq(metric_names[2]), ["2020", "2023"]]
    .stack()
    .between(1, 5)
    .all()
)
print("세 지표 결합 확인:", combined.shape)
display(combined)

## 10. 2023년 제주 공식값 비교

공식값은 가사 30.6%, 돌봄 43.5%, 사회경제적 지위 인식 2.61점이다. `계산값과 공식값의 차이`는 반올림 전 계산값에서 공식값을 뺀 값이며, 일치 여부는 지표별 지정 자릿수로 반올림한 최종값을 기준으로 판정한다. 보고서의 표준화·지표 가중 기여점수(1.700, 0.677, 1.764)와는 비교하지 않는다.

In [ ]:
JEJU_OFFICIAL = {
    "가사노동 공동 참여도": 30.6,
    "돌봄노동 공동 참여도": 43.5,
    "사회경제적 지위에 대한 인식": 2.61,
}

jeju_rows = []
for metric, official in JEJU_OFFICIAL.items():
    raw_value = raw_values[metric][2023]["제주"]
    final_value = final_tables[metric].loc[final_tables[metric]["지역"].eq("제주"), "2023"].iloc[0]
    jeju_rows.append(
        {
            "지표": metric,
            "반올림 전 계산값": raw_value,
            "최종 반올림값": final_value,
            "공식 측정값": official,
            "계산값과 공식값의 차이": raw_value - official,
            "일치 여부": bool(np.isclose(final_value, official)),
        }
    )

jeju_comparison = pd.DataFrame(jeju_rows)
assert jeju_comparison["일치 여부"].all()
display(jeju_comparison)

## 11. CSV 저장

지표별 18행 × 4열 CSV를 UTF-8 BOM(`utf-8-sig`)으로 저장한다. 값 열에는 `%` 또는 `점` 문자를 붙이지 않는다.

In [ ]:
saved_rows = []
for metric, path in OUTPUT_FILES.items():
    final_tables[metric].to_csv(path, index=False, encoding="utf-8-sig")
    saved_rows.append({"지표": metric, "경로": str(path), "크기(bytes)": path.stat().st_size})

saved_files = pd.DataFrame(saved_rows)
display(saved_files)

## 12. 저장 CSV 재로딩 및 최종 QA

In [ ]:
qa_rows = []
for metric, path in OUTPUT_FILES.items():
    loaded = pd.read_csv(path, encoding="utf-8-sig")
    expected = final_tables[metric]
    shape_ok = loaded.shape == (18, 4)
    columns_ok = loaded.columns.tolist() == ["지역", "세부지표", "2020", "2023"]
    region_order_ok = loaded["지역"].tolist() == REGION_ORDER
    indicator_ok = loaded["세부지표"].eq(metric).all()
    numeric_ok = all(pd.api.types.is_numeric_dtype(loaded[column]) for column in ["2020", "2023"])
    missing_preserved = int(loaded[["2020", "2023"]].isna().sum().sum()) == int(
        expected[["2020", "2023"]].isna().sum().sum()
    )
    bom_ok = path.read_bytes().startswith(b"\xef\xbb\xbf")
    pd.testing.assert_frame_equal(loaded, expected, check_dtype=True, check_exact=True)
    frame_equal = True
    assert (
        shape_ok
        and columns_ok
        and region_order_ok
        and indicator_ok
        and numeric_ok
        and missing_preserved
        and bom_ok
        and frame_equal
    )
    qa_rows.append(
        {
            "지표": metric,
            "크기": str(loaded.shape),
            "열 순서": columns_ok,
            "지역 순서": region_order_ok,
            "전체 프레임 일치": frame_equal,
            "숫자형": numeric_ok,
            "결측 보존": missing_preserved,
            "UTF-8 BOM": bom_ok,
        }
    )

current_signatures = {
    year: (path.stat().st_size, path.stat().st_mtime_ns) for year, path in SOURCE_FILES.items()
}
assert current_signatures == source_signatures, "원자료 크기 또는 수정시각이 변경됨"
assert all(len(data["시도코드"].unique()) == 17 for data in data_by_year.values())
assert all(table.shape == (18, 4) for table in final_tables.values())
assert combined.shape == (54, 4)
assert "2015" not in combined.columns

display(pd.DataFrame(qa_rows))
print("원자료 미변경:", current_signatures == source_signatures)
print("최종 QA 통과: 지표별 18×4, 결합 54×4, 17개 시도, 범위·순서·인코딩·숫자형 확인 완료")

## 13. 외부 기준값 검증

2023년 `전국` 행을 제외한 17개 시도의 **반올림 전 산출값**만 사용하여 제주 산출값, 제주 순위, 17개 시도 단순평균을 검증한다. 전국 원자료에서 직접 구한 가중값은 시도 단순평균에 포함하지 않는다.

- 순위: 내림차순 경쟁순위(`pandas.Series.rank(method='min', ascending=False)`)를 사용한다. 동점 지역에는 같은 최상위 순위를 부여하고 그 다음 순위는 동점 수만큼 건너뛴다(예: 1, 2, 2, 4).
- 단순평균: 17개 시도의 반올림 전 값을 산술평균한 뒤 비율은 소수점 첫째 자리, 점수는 소수점 둘째 자리에서 한 번만 반올림한다.
- 차이: `비교용 출력값 - 공식값`이다. 공식값은 검증에만 사용하며 산식이나 산출값을 조정하지 않는다.

In [ ]:
EXTERNAL_BENCHMARKS = {
    "가사노동 공동 참여도": {
        "제주 산출값": 30.6,
        "제주 순위": 4,
        "17개 시도 단순평균": 26.7,
    },
    "돌봄노동 공동 참여도": {
        "제주 산출값": 43.5,
        "제주 순위": 13,
        "17개 시도 단순평균": 47.4,
    },
    "사회경제적 지위에 대한 인식": {
        "제주 산출값": 2.61,
        "제주 순위": 4,
        "17개 시도 단순평균": 2.42,
    },
}
RANK_METHOD = "min"  # 경쟁순위: 동점은 같은 최상위 순위, 다음 순위는 건너뜀
external_validation_rows = []
national_vs_regional_rows = []

for metric in metric_names:
    regional_raw = pd.Series(raw_values[metric][2023], dtype="float64").drop(index="전국")
    regional_raw = regional_raw.reindex(REGION_ORDER[1:])
    assert regional_raw.index.tolist() == REGION_ORDER[1:]
    assert len(regional_raw) == 17 and regional_raw.notna().all()

    regional_rank = regional_raw.rank(method=RANK_METHOD, ascending=False)
    jeju_raw = float(regional_raw.loc["제주"])
    jeju_rank = int(regional_rank.loc["제주"])
    regional_mean_raw = float(regional_raw.mean())
    digits = ROUND_DIGITS[metric]

    calculated = {
        "제주 산출값": jeju_raw,
        "제주 순위": jeju_rank,
        "17개 시도 단순평균": regional_mean_raw,
    }
    comparison_values = {
        "제주 산출값": float(pd.Series([jeju_raw]).round(digits).iloc[0]),
        "제주 순위": jeju_rank,
        "17개 시도 단순평균": float(pd.Series([regional_mean_raw]).round(digits).iloc[0]),
    }

    for item, official in EXTERNAL_BENCHMARKS[metric].items():
        output_value = comparison_values[item]
        difference = output_value - official
        external_validation_rows.append(
            {
                "지표": metric,
                "검증 항목": item,
                "반올림 전 계산값": calculated[item],
                "비교용 출력값": output_value,
                "공식값": official,
                "차이": difference,
                "PASS 또는 FAIL": "PASS" if np.isclose(difference, 0.0) else "FAIL",
            }
        )

    national_vs_regional_rows.append(
        {
            "지표": metric,
            "전국 원자료 가중값(반올림 전)": raw_values[metric][2023]["전국"],
            "17개 시도 단순평균(반올림 전)": regional_mean_raw,
        }
    )

external_validation = pd.DataFrame(external_validation_rows)[
    [
        "지표",
        "검증 항목",
        "반올림 전 계산값",
        "비교용 출력값",
        "공식값",
        "차이",
        "PASS 또는 FAIL",
    ]
]

print(
    "동점 처리 방식: rank(method='min', ascending=False) 경쟁순위 — 동점은 같은 순위, 다음 순위는 건너뜀"
)
print("전국 원자료 가중값은 17개 시도 단순평균 계산에서 제외함")
with pd.option_context("display.precision", 12):
    display(external_validation)
    display(pd.DataFrame(national_vs_regional_rows))

failed_external_validation = external_validation.loc[
    external_validation["PASS 또는 FAIL"].eq("FAIL")
]
if failed_external_validation.empty:
    print("외부 기준값 검증: 9개 항목 모두 PASS")
else:
    print("외부 기준값 검증 FAIL — 아래 항목과 진단 정보를 확인")
    display(failed_external_validation)
    print("진단 1) 가중치: 가구원가중치 사용, 결측·0·음수 없음")
    print("진단 2) 유효응답: 가사·돌봄 1–3, 계층 1–5; 비해당·결측 제외")
    print("진단 3) 지역 범위: 전국 제외 17개 시도", regional_raw.index.tolist())
    print("진단 4) 반올림: 시도 원값으로 순위·평균 계산 후 지표별 마지막 1회 반올림")
    display(
        diagnostics.loc[
            (diagnostics["연도"].eq(2023)) & (diagnostics["지역"].isin(["전국", "제주"]))
        ]
    )